# **Deep Natural Language Processing @ PoliTO**


---
**Teaching Assistant:** Ali Yassine

**Credits:** Moreno La Quatra

**Practice 5:** Machine Translation - Part 2

## **Machine Translation**

Machine Translation is a sub-field of Natural Language Processing that aims at translating a text from a source language to a target language. In this practice, we will experiment with a Transformer-based model for Machine Translation. Specifically, we will benchmark the performance of a pre-trained MT model on Italian-English and English-Italian translation tasks.

![](https://www.deepl.com/img/press/desktop_ENIT_2020-01.png)

In this practice we will use a data collection provided by [tatoeba](https://tatoeba.org/). The following cell download a subset of the data collection, containing parallel Italian-English sentences.


### **Question 0: Parsing data**

The first step is to parse the data collection to generate a list of sentence pairs. The data are provided in `tsv` format, where each line contains a sentence pair in the following format:

`<source_language_sentence>\t<target_language_sentence>\n`


In [1]:
%%capture
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P6/train_it_en.tsv
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P6/test_it_en.tsv

In [2]:
import pandas as pd

df_mt_train = pd.read_csv("train_it_en.tsv", sep="\t", header=0, names=['it_id', 'it_sent', 'en_id', 'en_sent'])
df_mt_test  = pd.read_csv("test-it-en.tsv", sep="\t", header=0, names=['it_id', 'it_sent', 'en_id', 'en_sent'])

### **Question 1: Fine-tuning Seq2Seq model (IT->EN)**

One of the most common paradigm for modern transformer architectures is to use pre-training to learn a generic representation of the language and then fine-tune the model on a specific task. However, you can also fine-tune a model that is already pre-trained on a specific task to another dataset. In this question you will fine-tune a pre-trained MT model on a tatoeba dataset.

Exploit the [Trainer API](https://huggingface.co/transformers/training.html#fine-tuning-in-pytorch-with-the-trainer-api) to finetune and evaluate a [MarianMT](https://arxiv.org/pdf/1804.00344.pdf) sequence to sequence model for machine translation. The documentation for MarianMT is available [here](https://huggingface.co/transformers/model_doc/marian.html).

**Note 1:** select the pre-trained model according to the input-output pair (it-en)

**Note 2:** for the lab practice, please use a sub-set of the training data (e.g., 10000 sentences for training and 1000 sentences for validation)

In [ ]:
# your code here
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_id = "Helsinki-NLP/opus-mt-it-en"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
)
tokenizer.pad_token = tokenizer.eos_token

df_train = df_mt_train.iloc[:10000]
df_val = df_mt_train.iloc[10000:11000]

df_train = Dataset.from_pandas(df_train)
df_val = Dataset.from_pandas(df_val)

def preprocess(examples):
    inputs = tokenizer(
        examples['it_sent'],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    outputs = tokenizer(
        examples['en_sent'],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    inputs['labels'] = outputs['input_ids']

    return inputs

tokenized_train = df_train.map(
    preprocess,
    batched=True
)
tokenized_val = df_val.map(
    preprocess,
    batched=True
)

train_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=0.001,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=1e-2,
    logging_dir='./logs',
    report_to="none"
)
trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    args=train_args
)

trainer.train()

### **Question 2: Model evaluation**

Using the test set provided at the beginning of the practice, evaluate the fine-tuned model and report the BLEU and METEOR scores. How does the BLEU and METEOR scores compare with the one obtained in the previous question (Part 1 - Question 3)?

In [ ]:
# your code here
from tqdm import tqdm

df_test = Dataset.from_pandas(df_mt_test)
tokenized_test = df_test.map(
    preprocess,
    batched=True
)

# TODO: Find a way to use trainer.predict with 
"""
predictions = trainer.predict(
    tokenized_test
)
"""

model.eval()
batch_size = 4
decoded_preds = []
decoded_labels = []
for i in tqdm(range(0, len(tokenized_test), batch_size)):
    batch = tokenized_test[i:i+batch_size]
    
    inputs = {
        'input_ids': torch.tensor(batch['input_ids']).to(device),
        'attention_mask': torch.tensor(batch['attention_mask']).to(device)
    }
    
    with torch.no_grad():
        generated_tokens = model.generate(**inputs, max_length=128, num_beams=1)
    
    decoded_preds.extend(tokenizer.batch_decode(generated_tokens, skip_special_tokens=True))
    decoded_labels.extend(tokenizer.batch_decode(batch['labels'], skip_special_tokens=True))
    
    torch.cuda.empty_cache()

"""
preds, refs = [], []
for pred, ref in zip(predictions.predictions, predictions.label_ids):
    preds.append(tokenizer.decode(
            pred,
            skip_special_tokens=True
        )
    )
    refs.append(tokenizer.decode(
            ref,
            skip_special_tokens=True
        )
    )
"""

"""
decoded_preds = tokenizer.batch_decode(
    predictions.predictions,
    skip_special_tokens=True
)
decoded_labels = tokenizer.batch_decode(
    predictions.label_ids,
    skip_special_tokens=True
)
"""

In [ ]:
%%capture
!pip install sacrebleu evaluate nltk

In [ ]:
from sacrebleu import corpus_bleu
import evaluate

bleu_it_en = corpus_bleu(
    decoded_preds,
    [decoded_labels]
)
print(f"BLEU Score: {bleu_it_en.score}")

meteor_metric = evaluate.load("meteor")
meteor_score = meteor_metric.compute(
    predictions=decoded_preds,
    references=decoded_labels
)
print(f"METEOR Score: {meteor_score['meteor']}")

#### Comparison with Pretrained Model (Part 1, Question 3)

If you used OPUS-MT previously, you can compare the results:

**Reference results for IT → EN** (from Part 1 Q3):

- BLEU: 71.64  
- METEOR: 86.94  

Did the results improve compared to the pretrained model?

### **Question 3 (BONUS): Fine-tuning Seq2Seq model (EN->IT)**

Do the same fine-tuning procedure as in Question 1, but invert the translation direction: translate from English to Italian instead of Italian to English. Use the same Tatoeba dataset and a subset for training (e.g., 10,000 sentences) and validation (e.g., 1,000 sentences).

Tips / What to change compared to Q1:

Swap the source and target languages: inputs come from en_sent and labels come from it_sent.

Use the EN → IT pre-trained MarianMT model, e.g., "Helsinki-NLP/opus-mt-en-it".

Ensure that metric computations (BLEU, METEOR) reference the Italian sentences.

Everything else — dataset slicing, tokenization, training, evaluation — remains the same.


### **Question 4: Machine Translation using LLMs**

With the recent advent of LLMs, even the translation task can be addressed using such models.

LLMs have very broad general knowledge and are often able to express it using multiple languages. Thanks to this knowledge, acquired in the pretraining phase, it is possible to query an LLM without the need for further fine-tuning, even for a translation task.

After generating translations with the LLM, compute and report both BLEU and METEOR scores to quantitatively assess performance.

You can load an LLM from Hugging Face in the same way as a common transformer model.


In [ ]:
!pip install --upgrade transformers==4.50.0

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = 'microsoft/Phi-4-mini-instruct'

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True
).to('cuda')
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    padding_side="left"
)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

If you choose an instruction-tuned model, you can set up a conversation between the model and the user. You can also specify a "system" message to provide more details to the model as it should behave during the conversation.

In [4]:
from transformers import pipeline

system_message = "You are a helpful AI assistant that can translate sentences from English to Italian."
prompt = "Please translate the following English sentence in Italian language"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": f"{prompt}: Hi, how are you today?"},
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.001,
    "do_sample": True,
}

output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

Device set to use cuda


Ciao, come stai oggi?


Try to performe the machine translation task on the same dataset used in the previous exercise using this LLM, then evaluate the performance using the same metrics and compare the results.

In [6]:
# your code here
from datasets import Dataset
from tqdm import tqdm

df_test = Dataset.from_pandas(df_mt_test)

messages_list = []

for i in tqdm(range(len(df_test))):
    
    input_sentence = df_test[i]['en_sent']
    
    current_message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"{prompt}: {input_sentence}"} # Formattiamo direttamente
    ]
    
    messages_list.append(current_message)

outputs = pipe(
    messages_list,
    **generation_args,
    batch_size=256
)

100%|██████████| 5553/5553 [00:00<00:00, 13580.91it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 244.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 74.12 MiB is free. Process 19801 has 14.67 GiB memory in use. Of the allocated memory 13.03 GiB is allocated by PyTorch, and 1.51 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
predictions = []
for e in outputs:
    predictions.append(e[0]['generated_text'])

references = df_test['it_sent']

from sacrebleu import corpus_bleu
import evaluate

bleu_it_en = corpus_bleu(
    predictions,
    [references]
)
print(f"BLEU Score: {bleu_it_en.score}")

meteor_metric = evaluate.load("meteor")
meteor_score = meteor_metric.compute(
    predictions=predictions,
    references=references
)
print(f"METEOR Score: {meteor_score['meteor']}")